# 🤖 Google Colab X Gemma 4  AI Chat Assistant ( Google Colab TPU Edition)
### **High-Performance TPU-Optimized Colab Notebook**

This notebook is specifically configured and optimized to run Google's **Gemma 4 E4B** open-weights model natively on **Google Cloud TPUs (v5e / v4)** with `bfloat16` precision and persistent Google Drive memory.

### **Key Features:**
* ⚡ **TPU Native Acceleration:** Direct execution on Google Cloud TPU cores using `bfloat16` and native `device_map="tpu"`.
* 💾 **Google Drive Auto-Cache:** Model weights and chat logs are permanently saved to Google Drive (zero re-download on session restarts).
* 🧠 **Persistent Chat Memory:** Retains full conversational context across multi-turn interactions.
* 💬 **Continuous Live Chat:** Non-blocking in-notebook chat widget that runs continuously without session interruptions.

> **Prerequisite:** Make sure your Colab accelerator is set to TPU (**Runtime** ➡️ **Change runtime type** ➡️ **TPU**).

In [ ]:
# ==========================================================
# Step 1: Storage Setup & Google Cloud TPU Verification
# ==========================================================
import os
import sys
import torch

# 1. Mount Google Drive
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/Gemma_AI_Assistant"
    print("📁 Google Drive successfully connected!")
except Exception:
    BASE_DIR = "/content/Gemma_AI_Assistant"
    print("⚠️ Google Drive mount skipped. Using local workspace:", BASE_DIR)

MODEL_DIR = os.path.join(BASE_DIR, "model_cache")
HISTORY_DIR = os.path.join(BASE_DIR, "chat_history")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

# 2. Verify Google Cloud TPU
try:
    tpu_device = torch.device("tpu")
    _test = torch.ones(2, 2, device=tpu_device)
    print(f"✅ Google Cloud TPU Active! Verified Device: {_test.device}")
except Exception as e:
    print("❌ Google Cloud TPU not detected!")
    print("👉 Please enable TPU: Go to Runtime -> Change runtime type -> Select TPU -> Save")
    raise e

print("📁 Workspace initialized at:", BASE_DIR)

In [ ]:
# ==========================================================
# Step 2: Install Required Libraries
# ==========================================================
!pip install -q --upgrade transformers accelerate huggingface_hub ipywidgets
print("✅ Dependencies successfully installed!")

In [ ]:
# ==========================================================
# Step 3: Model Download & Persistent Cache
# ==========================================================
from huggingface_hub import snapshot_download

# Official Apache 2.0 open-weights Gemma 4 E4B Chat/Instruct Model
MODEL_ID = "google/gemma-4-E4B-it"

# Verify if model weights already exist in cache
is_downloaded = os.path.exists(MODEL_DIR) and len(os.listdir(MODEL_DIR)) > 3

if is_downloaded:
    print(f"✅ Model '{MODEL_ID}' is already cached in Google Drive!")
    print("📁 Cache Directory:", MODEL_DIR)
else:
    print(f"📥 Downloading official {MODEL_ID} to Google Drive...")
    print("Please wait, initial download takes approximately 2-4 minutes...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False
    )
    print("\n✅ Model successfully downloaded and cached permanently in Drive!")

In [ ]:
# ==========================================================
# Step 4: Stream Model Weights Directly onto TPU
# ==========================================================
from transformers import AutoProcessor, AutoModelForImageTextToText

print("⏳ Loading Gemma 4 directly onto Google Cloud TPU (bfloat16)...")

# Initialize Processor
processor = AutoProcessor.from_pretrained(MODEL_DIR)

# Load Model with native TPU device mapping
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_DIR,
    dtype=torch.bfloat16,
    device_map="tpu"
)

print("✅ Model successfully loaded on Google Cloud TPU!")

In [ ]:
# ==========================================================
# Step 5: Persistent Memory Manager & TPU Inference Engine
# ==========================================================
import json
from transformers import TextIteratorStreamer
from threading import Thread

HISTORY_FILE = os.path.join(HISTORY_DIR, "active_chat.json")

class MemoryManager:
    def __init__(self, filepath=HISTORY_FILE):
        self.filepath = filepath
        self.history = self.load()

    def load(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                return []
        return []

    def save(self):
        with open(self.filepath, "w", encoding="utf-8") as f:
            json.dump(self.history, f, ensure_ascii=False, indent=2)

    def add_message(self, role, content):
        self.history.append({"role": role, "content": content})
        self.save()

    def clear(self):
        self.history = []
        self.save()
        return "🧹 Conversation memory cleared!"

    def get_messages(self):
        return self.history

memory = MemoryManager()

def stream_chat_response(user_message, update_callback=None):
    memory.add_message("user", user_message)
    
    # Format conversation with canonical Gemma chat template
    formatted_prompt = processor.apply_chat_template(
        memory.get_messages(),
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = processor(text=formatted_prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    streamer = TextIteratorStreamer(
        processor.tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )
    
    # TPU-optimized greedy decoding parameters
    gen_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=250,
        do_sample=False,
        use_cache=True
    )
    
    thread = Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()
    
    full_response = ""
    for chunk in streamer:
        full_response += chunk
        if update_callback:
            update_callback(full_response)
            
    thread.join()
    memory.add_message("assistant", full_response.strip())
    return full_response.strip()

print("✅ TPU Inference & Memory Engine successfully initialized!")

In [ ]:
# ==========================================================
# Step 6: Interactive Continuous Chat (In-Notebook Widget)
# ==========================================================
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Scrollable chat stream
chat_output = widgets.Output(layout=widgets.Layout(height="450px", overflow_y="auto", padding="10px", border="1px solid #444"))

# Control bar
user_input = widgets.Text(placeholder="Type your message and press Enter...", layout=widgets.Layout(width="74%"))
send_btn = widgets.Button(description="Send 🚀", button_style="primary", layout=widgets.Layout(width="12%"))
clear_btn = widgets.Button(description="Clear Memory 🧹", button_style="danger", layout=widgets.Layout(width="14%"))

def render_history():
    with chat_output:
        clear_output()
        messages = memory.get_messages()
        if not messages:
            display(Markdown("*(No previous messages. Type a prompt below to start chatting!)*\n\n---"))
        for msg in messages:
            role = "👤 **You:**" if msg["role"] == "user" else "🤖 **Gemma 4:**"
            display(Markdown(f"{role}\n\n{msg['content']}\n\n---"))

def on_send(b=None):
    text = user_input.value.strip()
    if not text:
        return
    user_input.value = ""
    
    with chat_output:
        display(Markdown(f"👤 **You:**\n\n{text}\n\n---"))
        display(Markdown("**🤖 Gemma 4:**"))
        
        live_box = widgets.Output()
        display(live_box)
        
        def update_stream(live_text):
            with live_box:
                clear_output(wait=True)
                display(Markdown(live_text))
                
        with torch.inference_mode():
            stream_chat_response(text, update_stream)
            
        display(Markdown("\n\n---"))

def on_clear(b=None):
    memory.clear()
    render_history()

user_input.on_submit(on_send)
send_btn.on_click(on_send)
clear_btn.on_click(on_clear)

# Display UI Components
controls = widgets.HBox([user_input, send_btn, clear_btn])
display(widgets.VBox([chat_output, controls]))

# Load existing conversation from memory
render_history()